## TF-IDF Summarizer

**Objective :** Implements a statistical-based extractive summarizer using TF-IDF for long WikiHow documents (800+ words), and evaluate it using standard ROUGE metrics.

**Dataset:** WikiHow dataset, filtered to articles with 800+ words. A random sample of 500 articles is used, with the `text` column to be summarized by model and `headline` column serving as the reference summary.

**Steps:**
1. Load and clean the WikiHow dataset (drop nulls, remove noise from `text`/`headline` columns)
2. Tokenize each article into sentences using NLTK
3. Build a TF-IDF matrix per article and score each sentence by summing its TF-IDF values
4. Select the top 5 highest-scoring sentences (in original document order) as the summary
5. Evaluate generated summaries against reference headlines using ROUGE-1, ROUGE-2, and ROUGE-L (F-measure)

**Evaluation:** The generated summaries are evaluated using `rouge-score` library.

**Results (mean across 500 articles):**

| Metric | Score |
|---|---|
| ROUGE-1 | 0.250 |
| ROUGE-2 | 0.070 |
| ROUGE-L | 0.136 |

TF-IDF serves as the baseline — the simplest method, relying purely on term frequency without any semantic or structural understanding of the text.

In [2]:
import nltk
import sklearn
import sentence_transformers
import rouge_score

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
import pandas as pd

In [4]:
df = pd.read_csv('wikihowAll.csv')

In [5]:
df.describe()

,headline,title,text
count,214547,215364,214294
unique,214096,215364,209178
top,"\nAcquire a pot.,\nGather the ingredients need...",How to Be an Organized Artist1,",,"
freq,11,1,524


In [6]:
df.head()

,headline,title,text
0,"\nKeep related supplies in the same area.,\nMa...",How to Be an Organized Artist1,"If you're a photographer, keep all the necess..."
1,\nCreate a sketch in the NeoPopRealist manner ...,How to Create a Neopoprealist Art Work,See the image for how this drawing develops s...
2,"\nGet a bachelor’s degree.,\nEnroll in a studi...",How to Be a Visual Effects Artist1,It is possible to become a VFX artist without...
3,\nStart with some experience or interest in ar...,How to Become an Art Investor,The best art investors do their research on t...
4,"\nKeep your reference materials, sketches, art...",How to Be an Organized Artist2,"As you start planning for a project or work, ..."


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 215365 entries, 0 to 215364
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   headline  214547 non-null  str  
 1   title     215364 non-null  str  
 2   text      214294 non-null  str  
dtypes: str(3)
memory usage: 593.3 MB


In [8]:
df['title'].isnull().value_counts() 

title
False    215364
True          1
Name: count, dtype: int64

#### --> Non null count above shows that *'headline'* and *'text'* columns have some null values.
#### --> Drop null Values & Filter the dataframe for long articles.

In [ ]:
df.dropna(inplace=True)

In [11]:
df.info()

<class 'pandas.DataFrame'>
Index: 214294 entries, 0 to 215364
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   headline  214294 non-null  str  
 1   title     214294 non-null  str  
 2   text      214294 non-null  str  
dtypes: str(3)
memory usage: 594.9 MB


In [12]:
df['word count'] = df['text'].apply(lambda x: len(x.split()))

In [13]:
df['word count'].describe()

count    214294.000000
mean        439.283802
std         480.950694
min           0.000000
25%         139.000000
50%         299.000000
75%         538.000000
max       12118.000000
Name: word count, dtype: float64

Note: 75% of articles are below 538 words, 50% of articles are below 299 words, 25% of articles are below 139 words

In [14]:
len(df[df['word count'] > 800])
#df[df['word count'] > 800].count() --> Another way for same count.

30736

In [15]:
df_long = df[df['word count'] > 800].reset_index(drop=True)
# reset index --> replaces the old, broken(due to filtering) index with new, continuous index.
# drop=True : ensures that the old non-sequential(broken) index is completely discarded.

In [16]:
sample = df_long.sample(n=500, random_state=1)
# df.sample(frac=0.5, replace=True, random_state=1))....replace=True would mean that duplicate articles could start appearing since sampled datapoint 
# is put back and so it can be chosen again and thus duplicate which is not ideal for evaluating summarization.

 Note:  
 1. frac - fraction(50% - 0.5) of rows to return.[cannot be used with n]
 2. replace = False (By Default)

In [17]:
#print(sample['headline'].iloc[0])
print(sample['text'].iloc[0][:10000]) #slicing notation...the more you increase the number, the more words you will see from the text column.

 Meditating is a great way to relax your mind, and you can meditate almost anywhere and at any time. Just pick a quiet place where you can sit on level ground and close your eyes. Cross your legs and keep your hands on your lap. Focus on inhaling and exhaling, and let your body be governed by your breath. Keep as still as possible and avoid fidgeting.


Be aware of what you can't control. Focus and absorb the smells and sounds around you.
Clear your mind. Don't think about how much work you have left to do, or about what you're going to make for dinner. Just focus on clearing your mind and managing your breath.
Relax every part of your body. You can focus on one part of your body at a time until you feel that every part of you is loose and relaxed.

, Going out to the movies or watching a movie on television can help you escape into another universe and to take your mind off of your own problems. When you watch a movie, try to clear your mind as much as possible and think about what th

#### **Preprocessing Sample Dataset:**
1. In the text column, each paragraph is separated by commas(look for: , Going) and there is \n character in between paragraph to separate the paragraph.
2. These are noises and needs to be removed:  
--> Leading commas &  
--> White spaces / New line(\n\n)
3. .str.replace('find this', 'replace with this')


In [18]:
sample['text'] = sample['text'].str.replace('\n, ', ' ', regex=True) # deals with with leading commas like [, Going out] / The ','  that appears between sections is actually preceded by a newline — so we replace '\n,' - with a space
sample['text'] = sample['text'].str.replace('\n', ' ', regex=True) # remaining newline character with no commas i.e. space in between paragraph
sample['text'] = sample['text'].str.strip() #strip() removes any extra spaces and newline at the very start & end of each text.

In [19]:
print(sample['headline'].iloc[0])


Meditate.,
Watch a movie.,
Spend time with friends.,
Go for a long drive.,
Read.,
Calm your mind before bed.


In [20]:
sample['headline'] = sample['headline'].str.replace('\n', ' ', regex=True)
sample['headline'] = sample['headline'].str.replace('.,', '.', regex=True)
sample['headline'] = sample['headline'].str.strip()

In [21]:
print(sample['headline'].iloc[0])

Meditate. Watch a movie. Spend time with friends. Go for a long drive. Read. Calm your mind before bed.


--> Dataset is cleaned and Ready for Summarization Models!  
  
Tldr:
1. ✅ Data loaded
2. ✅ Nulls dropped
3. ✅ Filtered for 800+ word articles (selecting long documents)
4. ✅ Sampled 500 articles
5. ✅ Text and headline cleaned

#### -->**Tokenization:**

In [22]:
import nltk
nltk.download('punkt') #NLTK's Tokenizer.
nltk.download('stopwords')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\vivek\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vivek\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\vivek\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [23]:
sentences = nltk.sent_tokenize(sample['text'].iloc[0])

In [24]:
sentences[:3]

['Meditating is a great way to relax your mind, and you can meditate almost anywhere and at any time.',
 'Just pick a quiet place where you can sit on level ground and close your eyes.',
 'Cross your legs and keep your hands on your lap.']

In [25]:
sample['sentences'] = sample['text'].apply(lambda x : nltk.sent_tokenize(x))

In [26]:
sample.head()

,headline,title,text,word count,sentences
10685,Meditate. Watch a movie. Spend time with frien...,How to Relax and De Stress3,"Meditating is a great way to relax your mind, ...",883,"[Meditating is a great way to relax your mind,..."
13779,Define your interest. Determine your medium. D...,How to Become a Beauty Guru,"The beauty industry is a huge one, and it’s ev...",1242,"[The beauty industry is a huge one, and it’s e..."
16828,Relax. Make your move before you psyche yourse...,How to Ask for a Phone Number,If there's one single thing you can do to make...,2415,[If there's one single thing you can do to mak...
4999,Consider how old you will be when you want to ...,How to Buy a Hunting License in Pennsylvania,You are eligible to buy a hunting license so l...,1303,[You are eligible to buy a hunting license so ...
6426,Give less value to what other people think of ...,How to Live Free,"Other people are outside your control, and if ...",1393,"[Other people are outside your control, and if..."


# ----------TF-IDF Summarizer--------

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
vectorizer = TfidfVectorizer()

In [29]:
X = vectorizer.fit_transform(sample['sentences'].iloc[0])

In [30]:
vectorizer.get_feature_names_out() # this is only features name and hence it looks like a 1-D array.
# get_feature_names_out() is just 328 word strings.

array(['20', 'about', 'absorb', 'actually', 'afraid', 'after',
       'aggressively', 'all', 'allow', 'almost', 'aloud', 'an', 'and',
       'annoyed', 'another', 'anxious', 'any', 'anything', 'anywhere',
       'are', 'around', 'as', 'ask', 'at', 'avoid', 'aware', 'away',
       'back', 'bar', 'be', 'becomes', 'bed', 'before', 'blow', 'board',
       'body', 'break', 'breath', 'busy', 'but', 'by', 'calendar',
       'calmer', 'camomile', 'can', 'candles', 'catch', 'chance',
       'characters', 'choose', 'clear', 'clearing', 'close', 'coffee',
       'comedy', 'comfortable', 'commercials', 'concert', 'control',
       'couch', 'cross', 'crowded', 'crushed', 'cup', 'darken', 'day',
       'deciding', 'detail', 'dinner', 'distracted', 'do', 'doing', 'don',
       'down', 'drive', 'drives', 'driving', 'during', 'dvr', 'each',
       'easy', 'either', 'empowered', 'enough', 'environment', 'escape',
       'especially', 'even', 'events', 'every', 'exhaling', 'eyes',
       'fall', 'fantast

In [31]:
print(X.shape) #(number of sentences, number of unique words)
# confirms that X is a 2-D array.
# [n_samples, n_features]-->2D Array | [n_samples, ]-->1D Array.
# (no_rows, no_columns)

(51, 328)


In [32]:
X # TF-IDF Matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 772 stored elements and shape (51, 328)>

X is a Sparse Matrix, For TF-IDF calculations --> Convert it to Array.

X.toarray() --> convert a sparse matrix to a regular numpy array.

Note: np.array wouldn't work because most values in X are zero and sparse format don't store these zero's somehow and saves memory. But in Array you need those zero's.
np.array wouldn't include those zero's and so need to use X.toarray().  
[np.array - Used to create Array.]

In [33]:
X.toarray()

array([[0.        , 0.        , 0.        , ..., 0.        , 0.11919528,
        0.14304553],
       [0.        , 0.        , 0.        , ..., 0.        , 0.11476864,
        0.13773315],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.49477879],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]], shape=(51, 328))

In [34]:
sentence_scores = X.toarray().sum(axis=1) # Sum along columns[columns collapses and gives sum(scores) of each rows(sentences).]


In [35]:
sentence_scores.shape

(51,)

In [36]:
sentence_scores

array([3.97167919, 3.70639931, 2.70866491, 3.42345857, 2.53628909,
       2.53615195, 2.71874617, 1.6674946 , 4.11689793, 2.90920619,
       2.38482505, 4.06579653, 4.8437483 , 5.1692755 , 2.77227952,
       3.95375515, 3.05829572, 4.01575564, 3.57170302, 2.94196817,
       4.85152107, 4.32122078, 4.88560173, 3.90522951, 2.90217521,
       4.48141596, 1.41244972, 4.61524865, 2.89073446, 5.23392913,
       4.56607692, 5.16551428, 1.99750555, 4.49432327, 3.16253915,
       3.09029348, 5.16279166, 2.91855047, 4.75990795, 3.87428855,
       4.78473214, 3.732488  , 3.26030653, 4.9138983 , 3.00167577,
       2.57255336, 3.35982489, 3.28322626, 2.8919482 , 3.72983083,
       3.43840763])

In [37]:
top_indices = sentence_scores.argsort()[::-1][:5] # Sorts the values of array in ascending order(by default) using indices.
#[::-1] - Makes it descending.
top_indices
#argsort() - returns the indices that would sort an array(in ascending order).

array([29, 13, 31, 36, 43])

In [38]:
sorted_indices = sorted(top_indices)
sorted_indices

[np.int64(13), np.int64(29), np.int64(31), np.int64(36), np.int64(43)]

In [ ]:
sentences = sample['sentences'].iloc[0] # here 'sentences' holds the tokenized sentences of the first article(.iloc[0]).
summary = ' '.join(sentences[i] for i in sorted_indices)

In [41]:
print(summary) # Summary of only the first article.

When you watch a movie, try to clear your mind as much as possible and think about what the characters are doing and saying instead of what you're going to do or say after the movie. If you have a busy schedule, choose a board game night or go out to see a comedy with your friends, instead of going to a crowded bar where you won't have as much of a chance to laugh. You may feel overpowered by nasty traffic, or annoyed by how aggressively other people drive during the day, but if you take to the roads at night, you'll feel calmer and more empowered. If you're leaving a party after hours of laughing, sharing your feelings, and listening to your friends, taking a 20-minute solo drive before going home can help you wind down. If you feel so stressed out that you can't focus on what you're reading, take a break to meditate, or whisper the words aloud until you absorb their meaning.


### TD-IDF Summarizer Function:

In [42]:
def tfidf_summarizer(sentences):
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(sentences)
    sentence_scores = X.toarray().sum(axis=1)
    top_indices = sentence_scores.argsort()[::-1][:5]
    sorted_indices = sorted(top_indices)
    summary = ' '.join(sentences[i] for i in sorted_indices)
    return summary

In [43]:
sample['tfidf_summary'] = sample['sentences'].apply(tfidf_summarizer)

In [45]:
sample.head()

,headline,title,text,word count,sentences,tfidf_summary
10685,Meditate. Watch a movie. Spend time with frien...,How to Relax and De Stress3,"Meditating is a great way to relax your mind, ...",883,"[Meditating is a great way to relax your mind,...","When you watch a movie, try to clear your mind..."
13779,Define your interest. Determine your medium. D...,How to Become a Beauty Guru,"The beauty industry is a huge one, and it’s ev...",1242,"[The beauty industry is a huge one, and it’s e...",In this vibrant industry where trends are cons...
16828,Relax. Make your move before you psyche yourse...,How to Ask for a Phone Number,If there's one single thing you can do to make...,2415,[If there's one single thing you can do to mak...,Though it's always difficult (some might say a...
4999,Consider how old you will be when you want to ...,How to Buy a Hunting License in Pennsylvania,You are eligible to buy a hunting license so l...,1303,[You are eligible to buy a hunting license so ...,"Even though you are 11 years old, you would be..."
6426,Give less value to what other people think of ...,How to Live Free,"Other people are outside your control, and if ...",1393,"[Other people are outside your control, and if...","Better still, learn how to disarm such people ..."


### Evaluation:

In [46]:
from rouge_score import rouge_scorer

In [ ]:
# Intialize the scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

In [48]:
scores = scorer.score(sample['headline'].iloc[0], sample['tfidf_summary'].iloc[0])
scores

{'rouge1': Score(precision=0.07471264367816093, recall=0.6842105263157895, fmeasure=0.13471502590673579),
 'rouge2': Score(precision=0.017341040462427744, recall=0.16666666666666666, fmeasure=0.031413612565445025),
 'rougeL': Score(precision=0.05747126436781609, recall=0.5263157894736842, fmeasure=0.10362694300518134)}

In [49]:
sample['tfidf_score'] = sample.apply(lambda x: scorer.score(x['headline'], x['tfidf_summary']), axis=1)

In [50]:
sample['sentences']

10685    [Meditating is a great way to relax your mind,...
13779    [The beauty industry is a huge one, and it’s e...
16828    [If there's one single thing you can do to mak...
4999     [You are eligible to buy a hunting license so ...
6426     [Other people are outside your control, and if...
                               ...                        
21719    [If you want to build abdominal strength and f...
5529     [If you have not bought shoes for a while, con...
6055     [Warm water will open up your pores and help y...
21822    [Leafy greens are an integral part of a raw di...
13420    [In most cases, you must have a law degree to ...
Name: sentences, Length: 500, dtype: object

In [51]:
sample.head()

,headline,title,text,word count,sentences,tfidf_summary,tfidf_score
10685,Meditate. Watch a movie. Spend time with frien...,How to Relax and De Stress3,"Meditating is a great way to relax your mind, ...",883,"[Meditating is a great way to relax your mind,...","When you watch a movie, try to clear your mind...","{'rouge1': (0.07471264367816093, 0.68421052631..."
13779,Define your interest. Determine your medium. D...,How to Become a Beauty Guru,"The beauty industry is a huge one, and it’s ev...",1242,"[The beauty industry is a huge one, and it’s e...",In this vibrant industry where trends are cons...,"{'rouge1': (0.10344827586206896, 0.36206896551..."
16828,Relax. Make your move before you psyche yourse...,How to Ask for a Phone Number,If there's one single thing you can do to make...,2415,[If there's one single thing you can do to mak...,Though it's always difficult (some might say a...,"{'rouge1': (0.20652173913043478, 0.55882352941..."
4999,Consider how old you will be when you want to ...,How to Buy a Hunting License in Pennsylvania,You are eligible to buy a hunting license so l...,1303,[You are eligible to buy a hunting license so ...,"Even though you are 11 years old, you would be...","{'rouge1': (0.24914675767918087, 0.61344537815..."
6426,Give less value to what other people think of ...,How to Live Free,"Other people are outside your control, and if ...",1393,"[Other people are outside your control, and if...","Better still, learn how to disarm such people ...","{'rouge1': (0.15270935960591134, 0.36470588235..."


In [52]:
sample['tfidf_score'].iloc[0]['rouge1'].fmeasure

0.13471502590673579

In [53]:
sample['rouge1_f'] = sample['tfidf_score'].apply(lambda x : x['rouge1'].fmeasure)
sample['rouge2_f'] = sample['tfidf_score'].apply(lambda x : x['rouge2'].fmeasure)
sample['rougeL_f'] = sample['tfidf_score'].apply(lambda x : x['rougeL'].fmeasure)


In [54]:
print(sample[['rouge1_f', 'rouge2_f', 'rougeL_f']].mean()) # Baseline that will be compared with TextRank and BERT summarizers result.

rouge1_f    0.277661
rouge2_f    0.065701
rougeL_f    0.147591
dtype: float64


### TF-IDF summarizer is complete. ✅

1. Loaded and cleaned a 580MB dataset
2. Filtered and sampled articles
3. Built a TF-IDF summarizer completely from scratch
4. Evaluated it with ROUGE scores

In [55]:
sample.to_csv('sample.csv', index=False)
# Saving the sample dataframe.